# SWAN-SF Sliding-Window + Decomposition LSTM

In [1]:
# Configure project paths, dependencies, and hyperparameters
from pathlib import Path
import random
import numpy as np
import sys, os

sys.path.append(os.path.abspath("../scripts"))

from data_utils import find_project_root, load_sequence_data
from preprocessing import compute_feature_stats, normalize_sequence_data
from metrics import compute_metrics, sweep_thresholds, format_metrics
from losses import FocalLoss

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'Cleaned SWANSF Dataset'
print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)

# Sliding-window + training hyperparameters (mirroring 04_lstm_sliding_window)
WINDOW_SIZE = 32  # timesteps per window
WINDOW_STRIDE = 4
MAX_POS_WINDOWS_PER_SAMPLE = 16
MAX_NEG_WINDOWS_PER_SAMPLE = 12
MAX_NEG_TO_POS_RATIO = 1.5
POSITIVE_WINDOW_TAIL_FRACTION = 0.5
NEGATIVE_WINDOW_SAMPLE_FRACTION = 0.5
AGGREGATION_MODE = 'max'
AGGREGATION_PERCENTILE = 0.9
MAX_ACCEPTABLE_FPR = 0.05
THRESHOLD_GRID = np.linspace(0.2, 0.9, 29)
BATCH_SIZE = 128
NUM_EPOCHS = 25
EARLY_STOP_PATIENCE = 5
LR_SCHEDULER_PATIENCE = 2
LR_SCHEDULER_FACTOR = 0.5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
# Trend smoothing kernel size for decomposition block
TREND_KERNEL_SIZE = 5

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
rng = np.random.default_rng(random_seed)

Project root: /Users/chintan/Drive/Projects/solar-flare-classification
Data dir: /Users/chintan/Drive/Projects/solar-flare-classification/data/Cleaned SWANSF Dataset


In [2]:
# Import PyTorch modules
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Restrict modeling to the top-5 features identified in EDA
SELECTED_FEATURES = ('SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ')

print('Torch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', device)
print('Num selected attributes:', len(SELECTED_FEATURES))

Torch version: 2.11.0
Using device: mps
Num selected attributes: 5


In [3]:
# Load partitions, normalize features, and create train/val/test splits
X_train_all, y_train_all, X_val_all, y_val_all, X_test_all, y_test_all = load_sequence_data(
    DATA_DIR,
    SELECTED_FEATURES,
    )

print('Train shape (concat):', X_train_all.shape)
print('Validation shape:', X_val_all.shape)
print('Test shape (concat):', X_test_all.shape)

feature_mean, feature_std = compute_feature_stats(X_train_all)
print('Feature mean (first 5):', feature_mean[:5])
print('Feature std (first 5):', feature_std[:5])

X_train_all = normalize_sequence_data(X_train_all, feature_mean, feature_std)
X_val_all = normalize_sequence_data(X_val_all, feature_mean, feature_std)
X_test_all = normalize_sequence_data(X_test_all, feature_mean, feature_std)

Train shape (concat): (77865, 60, 5)
Validation shape: (19899, 60, 5)
Test shape (concat): (331185, 60, 5)
Feature mean (first 5): [0.79975486 0.637749   0.6846829  0.2945301  0.8939753 ]
Feature std (first 5): [0.11830382 0.18934865 0.17060669 0.10897669 0.1079974 ]


In [4]:
# Sliding window utilities and PyTorch dataset wrappers
from typing import Dict, Optional


def build_window_index(
    X: np.ndarray,
    y: np.ndarray,
    window_size: int,
    stride: int,
    max_pos_windows: Optional[int] = None,
    max_neg_windows: Optional[int] = None,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, np.ndarray]:
    if rng is None:
        rng = np.random.default_rng()

    sample_indices = []
    window_offsets = []
    labels = []
    sample_ids = []

    for sample_idx, (seq, label) in enumerate(zip(X, y)):
        if seq.shape[0] < window_size:
            continue
        starts = list(range(0, seq.shape[0] - window_size + 1, stride))
        if not starts:
            continue

        # Restrict positive windows to the tail where flare signatures are more likely to appear.
        if label == 1 and 0 < POSITIVE_WINDOW_TAIL_FRACTION < 1.0:
            tail_count = max(1, int(len(starts) * POSITIVE_WINDOW_TAIL_FRACTION))
            starts = starts[-tail_count:]
        # Randomly subsample negative windows early to avoid over-representing quiet sequences.
        if label == 0 and 0 < NEGATIVE_WINDOW_SAMPLE_FRACTION < 1.0:
            keep_count = max(1, int(len(starts) * NEGATIVE_WINDOW_SAMPLE_FRACTION))
            keep_idx = rng.choice(len(starts), size=keep_count, replace=False)
            starts = [starts[i] for i in keep_idx]

        limit = max_pos_windows if label == 1 else max_neg_windows
        if limit is not None and len(starts) > limit:
            keep_idx = rng.choice(len(starts), size=limit, replace=False)
            starts = [starts[i] for i in keep_idx]

        sample_indices.extend([sample_idx] * len(starts))
        window_offsets.extend(starts)
        labels.extend([float(label)] * len(starts))
        sample_ids.extend([sample_idx] * len(starts))

    if not sample_indices:
        raise ValueError('No windows were generated; check window size/stride settings.')

    return {
        'sample_indices': np.array(sample_indices, dtype=np.int64),
        'window_offsets': np.array(window_offsets, dtype=np.int32),
        'labels': np.array(labels, dtype=np.float32),
        'sample_ids': np.array(sample_ids, dtype=np.int64),
    }


def rebalance_window_index(
    data: Dict[str, np.ndarray],
    max_neg_to_pos_ratio: Optional[float],
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, np.ndarray]:
    if max_neg_to_pos_ratio is None:
        return data
    if rng is None:
        rng = np.random.default_rng()

    labels = data['labels']
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return data

    allowed_neg = int(len(pos_idx) * max_neg_to_pos_ratio)
    if allowed_neg <= 0 or len(neg_idx) <= allowed_neg:
        return data

    keep_neg_idx = rng.choice(neg_idx, size=allowed_neg, replace=False)
    keep_idx = np.concatenate([pos_idx, keep_neg_idx])
    keep_idx.sort()

    return {
        key: value[keep_idx]
        for key, value in data.items()
    }


def summarize_index(name: str, data: Dict[str, np.ndarray]):
    total = len(data['labels'])
    pos = int(data['labels'].sum())
    neg = total - pos
    ratio = neg / pos if pos > 0 else float('inf')
    print(
        f'{name}: windows={total:,} | pos={pos} ({pos/total:.2%}) | '
        f'neg={neg} ({neg/total:.2%}) | neg/pos={ratio:.2f}'
    )


class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, metadata: Dict[str, np.ndarray], window_size: int):
        self.X = X
        self.sample_indices = metadata['sample_indices']
        self.window_offsets = metadata['window_offsets']
        self.labels = torch.from_numpy(metadata['labels']).float()
        self.sample_ids = torch.from_numpy(metadata['sample_ids']).long()
        self.window_size = window_size

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample_idx = self.sample_indices[idx]
        start = self.window_offsets[idx]
        seq = self.X[sample_idx]
        window = seq[start : start + self.window_size]
        window_tensor = torch.as_tensor(window, dtype=torch.float32)
        return window_tensor, self.labels[idx], self.sample_ids[idx]


def make_dataloaders(window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    train_meta = build_window_index(
        X_train_all,
        y_train_all,
        window_size,
        stride,
        max_pos_windows=MAX_POS_WINDOWS_PER_SAMPLE,
        max_neg_windows=MAX_NEG_WINDOWS_PER_SAMPLE,
        rng=rng,
    )
    val_meta = build_window_index(
        X_val_all,
        y_val_all,
        window_size,
        stride,
        max_pos_windows=MAX_POS_WINDOWS_PER_SAMPLE,
        max_neg_windows=MAX_NEG_WINDOWS_PER_SAMPLE,
        rng=rng,
    )
    test_meta = build_window_index(
        X_test_all,
        y_test_all,
        window_size,
        stride,
        max_pos_windows=None,
        max_neg_windows=None,
        rng=rng,
    )

    train_meta = rebalance_window_index(train_meta, MAX_NEG_TO_POS_RATIO, rng=rng)
    val_meta = rebalance_window_index(val_meta, MAX_NEG_TO_POS_RATIO, rng=rng)

    summarize_index('Train', train_meta)
    summarize_index('Val', val_meta)
    summarize_index('Test', test_meta)

    train_loader = DataLoader(WindowDataset(X_train_all, train_meta, window_size), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(WindowDataset(X_val_all, val_meta, window_size), batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader = DataLoader(WindowDataset(X_test_all, test_meta, window_size), batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    return train_loader, val_loader, test_loader, train_meta, val_meta, test_meta


train_loader, val_loader, test_loader, train_data, val_data, test_data = make_dataloaders()
train_loader, val_loader, test_loader

Train: windows=311,460 | pos=151492 (48.64%) | neg=159968 (51.36%) | neg/pos=1.06
Val: windows=79,596 | pos=39600 (49.75%) | neg=39996 (50.25%) | neg/pos=1.01
Test: windows=1,324,740 | pos=24936 (1.88%) | neg=1299804 (98.12%) | neg/pos=52.13


(<torch.utils.data.dataloader.DataLoader at 0x1130e3770>,
 <torch.utils.data.dataloader.DataLoader at 0x1175d2fd0>)

In [6]:
# Decomposition LSTM classifier for windows
class DecompositionWindowLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, dropout: float, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        # Depthwise 1D conv for per-feature smoothing (trend component)
        self.trend_conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=input_dim,
            kernel_size=kernel_size,
            padding=padding,
            groups=input_dim,
            bias=False,
        )
        with torch.no_grad():
            self.trend_conv.weight.fill_(1.0 / kernel_size)
        # LSTM over concatenated trend + residual
        self.lstm = nn.LSTM(
            input_size=input_dim * 2,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x_perm = x.permute(0, 2, 1)  # (batch, input_dim, seq_len)
        trend = self.trend_conv(x_perm)  # (batch, input_dim, seq_len)
        trend = trend.permute(0, 2, 1)   # (batch, seq_len, input_dim)
        resid = x - trend
        decomp = torch.cat([trend, resid], dim=-1)  # (batch, seq_len, 2 * input_dim)
        outputs, (h_n, _) = self.lstm(decomp)
        h_last = h_n[-1]  # (batch, hidden_dim)
        logits = self.fc(self.dropout(h_last)).squeeze(1)
        return logits


input_dim = len(SELECTED_FEATURES)
model = DecompositionWindowLSTM(input_dim, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, TREND_KERNEL_SIZE).to(device)
model

DecompositionWindowLSTM(
  (trend_conv): Conv1d(5, 5, kernel_size=(5,), stride=(1,), padding=(2,), groups=5, bias=False)
  (lstm): LSTM(10, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [10]:
# Training and evaluation utilities
from collections import defaultdict


pos_count = train_data['labels'].sum()
neg_count = len(train_data['labels']) - pos_count
pos_weight_value = neg_count / pos_count if pos_count > 0 else 1.0
print(f'Training windows: pos={int(pos_count)}, neg={int(neg_count)}, pos_weight={pos_weight_value:.2f}')

# Choose ONE loss (uncomment what you want)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value))  # BCE baseline
#criterion = FocalLoss(alpha=0.75, gamma=2.0)  # Focal Loss

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_SCHEDULER_FACTOR, patience=LR_SCHEDULER_PATIENCE
)


def get_current_lr(opt):
    return opt.param_groups[0]['lr']


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for windows, labels, _ in loader:
        windows = windows.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(windows)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * windows.size(0)
    return total_loss / len(loader.dataset)


def predict_scores(model, loader):
    model.eval()
    probs = []
    labels = []
    sample_ids = []
    with torch.no_grad():
        for windows, batch_labels, batch_ids in loader:
            windows = windows.to(device)
            logits = model(windows)
            batch_probs = torch.sigmoid(logits).cpu().numpy()
            probs.append(batch_probs)
            labels.append(batch_labels.numpy())
            sample_ids.append(batch_ids.numpy())
    probs = np.concatenate(probs)
    labels = np.concatenate(labels)
    sample_ids = np.concatenate(sample_ids)
    return labels, probs, sample_ids


def aggregate_scores(scores: list) -> float:
    # Aggregate multiple window scores into one sample-level probability.
    if AGGREGATION_MODE == 'max':
        return float(np.max(scores))
    if AGGREGATION_MODE == 'percentile':
        return float(np.quantile(scores, AGGREGATION_PERCENTILE))
    if AGGREGATION_MODE == 'median':
        return float(np.median(scores))
    return float(np.mean(scores))


def aggregate_by_sample(labels, probs, sample_ids):
    # Group window-level predictions by NOAA sample id before final evaluation.
    score_dict = defaultdict(list)
    label_dict = {}
    for label, prob, sid in zip(labels, probs, sample_ids):
        score_dict[int(sid)].append(prob)
        label_dict[int(sid)] = label
    agg_labels = []
    agg_scores = []
    for sid in sorted(label_dict.keys()):
        agg_labels.append(label_dict[sid])
        agg_scores.append(aggregate_scores(score_dict[sid]))
    return np.array(agg_labels), np.array(agg_scores)


def evaluate(model, loader, aggregate=True):
    labels, probs, sample_ids = predict_scores(model, loader)
    if aggregate:
        labels, probs = aggregate_by_sample(labels, probs, sample_ids)
    return labels, probs

Training windows: pos=151492, neg=159968, pos_weight=1.06


In [11]:
# Train decomposition LSTM with validation monitoring and early stopping
import copy

best_state = None
best_val_metrics = {'tss': -np.inf}
best_threshold = 0.5
history = []
epochs_without_improvement = 0
improvement_epsilon = 1e-4

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_labels_epoch, val_probs_epoch = evaluate(model, val_loader, aggregate=True)
    val_best, _ = sweep_thresholds(val_labels_epoch, val_probs_epoch, max_fpr=MAX_ACCEPTABLE_FPR)

    improved = val_best['tss'] > best_val_metrics.get('tss', -np.inf) + improvement_epsilon
    if improved:
        best_state = copy.deepcopy(model.state_dict())
        best_val_metrics = val_best
        best_threshold = val_best['threshold']
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    current_lr = get_current_lr(optimizer)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_tss': val_best['tss'],
        'val_fpr': val_best['fpr'],
        'threshold': val_best['threshold'],
        'lr': current_lr,
    })
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_tss={val_best['tss']:.4f} | threshold={val_best['threshold']:.2f} | val_fpr={val_best['fpr']:.3f} | lr={current_lr:.2e}")

    scheduler.step(val_best['tss'])

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(
            f"Early stopping triggered after {epoch} epochs (no val TSS improvement for {EARLY_STOP_PATIENCE} epochs).",
        )
        break

if best_state is not None:
    model.load_state_dict(best_state)

print('Best validation metrics:', format_metrics(best_val_metrics))
print('Best threshold:', round(float(best_threshold), 4))

Epoch 01 | loss=0.1228 | val_tss=0.8957 | threshold=0.60 | val_fpr=0.049 | lr=1.00e-03
Epoch 02 | loss=0.1087 | val_tss=0.8994 | threshold=0.45 | val_fpr=0.048 | lr=1.00e-03
Epoch 03 | loss=0.0996 | val_tss=0.8879 | threshold=0.45 | val_fpr=0.045 | lr=1.00e-03
Epoch 04 | loss=0.0940 | val_tss=0.8996 | threshold=0.35 | val_fpr=0.048 | lr=1.00e-03
Epoch 05 | loss=0.0885 | val_tss=0.9060 | threshold=0.35 | val_fpr=0.046 | lr=1.00e-03
Epoch 06 | loss=0.0837 | val_tss=0.8953 | threshold=0.35 | val_fpr=0.045 | lr=1.00e-03
Epoch 07 | loss=0.0780 | val_tss=0.8935 | threshold=0.20 | val_fpr=0.045 | lr=1.00e-03
Epoch 08 | loss=0.0731 | val_tss=0.8565 | threshold=0.25 | val_fpr=0.046 | lr=1.00e-03
Epoch 09 | loss=0.0636 | val_tss=0.8879 | threshold=0.20 | val_fpr=0.049 | lr=5.00e-04
Epoch 10 | loss=0.0603 | val_tss=0.8676 | threshold=0.25 | val_fpr=0.050 | lr=5.00e-04
Early stopping triggered after 10 epochs (no val TSS improvement for 5 epochs).
Best validation metrics: {'threshold': 0.35, 'roc_

In [13]:
# Evaluate on validation (for record) and test set using best validation threshold
import pandas as pd

val_labels_final, val_probs_final = evaluate(model, val_loader, aggregate=True)
val_metrics = compute_metrics(val_labels_final, val_probs_final, threshold=best_threshold)
test_labels, test_probs = evaluate(model, test_loader, aggregate=True)
test_metrics = compute_metrics(test_labels, test_probs, threshold=best_threshold)
test_best_under_constraint, _ = sweep_thresholds(test_labels, test_probs, max_fpr=MAX_ACCEPTABLE_FPR)

print(f'Aggregation mode: {AGGREGATION_MODE}')

rows = [
    ('Best validation sweep', best_val_metrics),
    ('Validation @ best threshold', val_metrics),
    ('Test @ best threshold', test_metrics),
    ('Test sweep (constrained) best', test_best_under_constraint),
]

table_rows = []
for label, metrics in rows:
    table_rows.append(
        {
            'Split / Setting': label,
            'Threshold': float(metrics['threshold']),
            'ROC-AUC': float(metrics['roc_auc']),
            'PR-AUC': float(metrics['pr_auc']),
            'TPR': float(metrics['tpr']),
            'FPR': float(metrics['fpr']),
            'TSS': float(metrics['tss']),
            'HSS': float(metrics['hss']),
        }
    )

metrics_table = pd.DataFrame(table_rows).round(4)
metrics_table

# Save sliding-window decomposition LSTM test probabilities and labels for ensemble notebook
import os
os.makedirs("../outputs", exist_ok=True)

y_test = test_labels
np.save("../outputs/sliding_decomposition_lstm_probs.npy", test_probs)
np.save("../outputs/y_test.npy", y_test)
print("Saved shape:", test_probs.shape)

Aggregation mode: max
Saved shape: (331185,)
